# ResQ Hybrid Model Training on Google Colab
This notebook will extract your dataset, install the required libraries, and train your YOLOv8 model.

### Step 1: Upload the Data
On the left sidebar, click the **Folder icon**, then click the **Upload** button.

Upload the `combined_dataset.zip` file from your `resq_backend` folder.

In [ ]:
!pip install ultralytics==8.3.244
import torch
if torch.cuda.is_available():
    print("GPU is enabled and ready to go!")
else:
    print("WARNING: GPU is not enabled! Go to Runtime -> Change runtime type -> Hardware accelerator -> select T4 GPU.")

In [ ]:
import zipfile
import os

# Make sure you uploaded combined_dataset.zip to the Colab file explorer first!
zip_path = 'combined_dataset.zip'
extract_path = 'dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print('Dataset extracted successfully! Ready for training.')
else:
    print('ERROR: Please upload combined_dataset.zip to the left sidebar first! It might still be uploading if you just started it.')

In [ ]:
from ultralytics import YOLO

# Fix paths in dataset.yaml to be absolute for Colab's Linux environment
yaml_path = 'dataset/combined_dataset/dataset.yaml'
with open(yaml_path, 'r') as file:
  yaml_data = file.read()

# Replace windows/relative paths with the Colab absolute path
yaml_data = yaml_data.replace('path: D:/PROJECTS/ResQ/resq_backend/data/combined_dataset', 'path: /content/dataset/combined_dataset')
yaml_data = yaml_data.replace('path: ../data/combined_dataset', 'path: /content/dataset/combined_dataset')
yaml_data = yaml_data.replace('path: ./data/combined_dataset', 'path: /content/dataset/combined_dataset')

with open(yaml_path, 'w') as file:
  file.write(yaml_data)

print('Fixed YAML paths for Colab.')

In [ ]:
# Load the model and start training on the GPU
model = YOLO('yolov8m-pose.pt')

print("Starting Model Training... This will take a while!")
model.train(data=yaml_path, epochs=50, imgsz=1280, batch=16, device=0)

In [ ]:
# Download the trained weights automatically when done!
from google.colab import files
import os

weights_path = 'runs/pose/train/weights/best.pt'
if os.path.exists(weights_path):
    files.download(weights_path)
else:
    # Sometimes training runs are sequentially numbered (train2, train3, etc.)
    print('Checking for alternative output folders...')
    import glob
    paths = glob.glob('runs/pose/train*/weights/best.pt')
    if paths:
        files.download(paths[-1])
        print(f'Downloading {paths[-1]}')
    else:
        print('Training might not have finished or weights could not be found.')